# pinball_endpoint

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Each setting's agents: the large buffer against three agents at the setting's
# memory budget.
SETTINGS = {
    "medium": {
        "Large recency (10k)": "ddqn_pinball",
        "Small recency (1k)": "ddqn_medium_pinball",
        "Small recency (1k) 10-step": "ddqn_medium_nstep_pinball",
        "Endpoint replay (1k)": "endpoint_pinball",
    },
    "small": {
        "Large recency (10k)": "ddqn_pinball",
        "Small recency (500)": "ddqn_small_pinball",
        "Small recency (500) 10-step": "ddqn_small_nstep_pinball",
        "Endpoint replay (500)": "endpoint_small_pinball",
    },
}

In [ ]:
# Load each component's stored runs once and compute its mean return over seeds
# with a 95% bootstrap CI.
names = {name for agents in SETTINGS.values() for name in agents.values()}
curves = {}
for name in names:
    runs = load_results(EXPERIMENT, name)
    if runs is not None:
        timesteps, returns = subsample(return_curve(runs.reward, runs.done))
        curves[name] = (timesteps, mean_ci(returns), len(returns))

In [ ]:
# Style for the learning-curve plots.
STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
    "legend.loc": "lower right",
}
COLORS = {
    "ddqn_pinball": "black",
    "ddqn_medium_pinball": "tab:blue",
    "ddqn_medium_nstep_pinball": "tab:purple",
    "endpoint_pinball": "tab:red",
    "ddqn_small_pinball": "tab:blue",
    "ddqn_small_nstep_pinball": "tab:purple",
    "endpoint_small_pinball": "tab:red",
}
BAND_ALPHA = 0.2
LEGEND_LABEL = "{agent} (n={runs})"
TITLES = {
    "medium": "Pinball easy, 1k memory  (mean ± 95% bootstrap CI)",
    "small": "Pinball easy, 500 memory  (mean ± 95% bootstrap CI)",
}
XLABEL = "Timestep"
YLABEL = "Return"
YLABEL_KW = {"rotation": 0, "ha": "right", "va": "center", "labelpad": 12}
YLIM = (-1000, 0)

In [ ]:
# Plot each setting's mean returns and CI bands over time.
figures = {}
for setting, agents in SETTINGS.items():
    drawn = {agent: name for agent, name in agents.items() if name in curves}
    if not drawn:
        continue
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for agent, name in drawn.items():
            timesteps, (mean, low, high), runs = curves[name]
            color = COLORS[name]
            ax.fill_between(timesteps, low, high, color=color, alpha=BAND_ALPHA)
            label = LEGEND_LABEL.format(agent=agent, runs=runs)
            ax.plot(timesteps, mean, color=color, label=label)
        ax.set(title=TITLES[setting], xlabel=XLABEL, ylim=YLIM)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.legend()
        fig.tight_layout()
    figures[f"curves_{setting}"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for suffix in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{suffix}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")